# Diabetes Prediction Project

End-to-end machine learning project on the **diabetes.csv** dataset.

Each task below is implemented one by one:
1. Problem statement, dataset summary and initial observations
2. Data preprocessing (missing values, outliers, encoding, scaling) and EDA
3. Algorithm selection: implementation with a library (scikit-learn) and from scratch (without library)
4. Testing on the test set, evaluation metrics and overfitting / underfitting check
5. Advanced models, cross-validation, model comparison and hyperparameter tuning

*(Tasks 6 and 7 - frontend and deployment - are intentionally not included here.)*

## Task 1: Problem Statement, Dataset Summary and Initial Observations

### Problem Statement

Given diagnostic medical measurements of patients (glucose level, blood pressure, BMI, age, etc.), the goal is to **predict whether a patient is diabetic or not**. This is a **binary classification** problem where the target column `Outcome` takes the values `Diabetic` and `Non Diabetic`.

The trained model can act as a decision-support tool to flag high-risk patients for early screening.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

sns.set(style='whitegrid')
%matplotlib inline

In [ ]:
# Load the dataset
df = pd.read_csv('diabetes.csv')
df.head()

In [ ]:
# Shape of the dataset (rows, columns)
print('Rows, Columns:', df.shape)

In [ ]:
# Column names and data types
df.info()

In [ ]:
# Statistical summary of numerical features
df.describe()

In [ ]:
# Target distribution
print(df['Outcome'].value_counts())
print()
print(df['Outcome'].value_counts(normalize=True) * 100)

### Initial Observations

- The dataset has **768 rows** and **9 columns** (8 features + 1 target `Outcome`).
- All feature columns are numeric; `Outcome` is categorical (`Diabetic` / `Non Diabetic`).
- Columns such as `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin` and `BMI` contain **0 values**, which are physiologically impossible and actually represent **missing values** to be handled in Task 2.
- The classes are **imbalanced** (roughly 65% Non Diabetic vs 35% Diabetic), which we keep in mind during evaluation.

## Task 2: Data Preprocessing and EDA

Handle missing values, detect and handle outliers, encode the categorical target, scale numerical features, and perform exploratory data analysis.

In [ ]:
# --- Handle Missing Values ---
# In this dataset a 0 in these medical columns is not valid and represents a missing value.
cols_with_zero_missing = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

# Explicit NaN check first
print('Explicit NaN counts:')
print(df.isnull().sum())

# Count the disguised (zero) missing values
print('\nZero (disguised missing) counts:')
print((df[cols_with_zero_missing] == 0).sum())

In [ ]:
# Replace the invalid zeros with NaN, then impute with the median (robust to outliers)
df[cols_with_zero_missing] = df[cols_with_zero_missing].replace(0, np.nan)

for col in cols_with_zero_missing:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

print('Missing values after imputation:')
print(df.isnull().sum())

In [ ]:
# --- Identify and Handle Outliers (IQR method) ---
numeric_cols = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
                'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

# Visualise outliers with boxplots before capping
plt.figure(figsize=(14, 8))
for i, col in enumerate(numeric_cols):
    plt.subplot(2, 4, i + 1)
    sns.boxplot(y=df[col])
    plt.title(col)
plt.tight_layout()
plt.show()

In [ ]:
# Cap outliers using the IQR rule (winsorization) instead of dropping rows,
# so we keep all 768 samples.
def cap_outliers_iqr(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return series.clip(lower, upper), lower, upper

for col in numeric_cols:
    capped, low, high = cap_outliers_iqr(df[col])
    n_out = ((df[col] < low) | (df[col] > high)).sum()
    df[col] = capped
    print(f'{col}: capped {n_out} outliers to [{low:.2f}, {high:.2f}]')

In [ ]:
# --- Encode the Categorical Target ---
# Outcome is categorical: 'Diabetic' / 'Non Diabetic' -> 1 / 0
df['Outcome'] = df['Outcome'].map({'Non Diabetic': 0, 'Diabetic': 1})
print(df['Outcome'].value_counts())
df.head()

In [ ]:
# --- Exploratory Data Analysis (EDA) ---
# Distribution of each numeric feature
df[numeric_cols].hist(figsize=(14, 10), bins=20)
plt.suptitle('Feature Distributions')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix')
plt.show()

In [ ]:
# Class balance and a couple of feature-vs-target relationships
plt.figure(figsize=(14, 4))

plt.subplot(1, 3, 1)
sns.countplot(x='Outcome', data=df)
plt.title('Class Balance (0=Non Diabetic, 1=Diabetic)')

plt.subplot(1, 3, 2)
sns.boxplot(x='Outcome', y='Glucose', data=df)
plt.title('Glucose vs Outcome')

plt.subplot(1, 3, 3)
sns.boxplot(x='Outcome', y='BMI', data=df)
plt.title('BMI vs Outcome')

plt.tight_layout()
plt.show()

In [ ]:
# --- Normalize / Scale Numerical Features ---
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X = df[numeric_cols]
y = df['Outcome']

# Split first, then scale using training statistics only (avoids data leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Train shape:', X_train_scaled.shape)
print('Test shape:', X_test_scaled.shape)
print('Feature means after scaling (train ~0):', np.round(X_train_scaled.mean(axis=0), 2))

## Task 3: Algorithm Selection

**Chosen algorithm: Logistic Regression.**

For a binary medical-diagnosis problem with numeric features, Logistic Regression is a strong, interpretable baseline. It outputs calibrated probabilities and its coefficients show the direction and strength of each feature's effect. We first train it with a **library (scikit-learn)** and then re-implement it **from scratch (without any ML library)** using gradient descent.

In [ ]:
# --- 3a. Logistic Regression using a library (scikit-learn) ---
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

lr_lib = LogisticRegression(max_iter=1000)
lr_lib.fit(X_train_scaled, y_train)

y_pred_lib = lr_lib.predict(X_test_scaled)
print('Library Logistic Regression - Test Accuracy:', round(accuracy_score(y_test, y_pred_lib), 4))

In [ ]:
# --- 3b. Logistic Regression from scratch (without any ML library) ---
# Implemented with pure NumPy: sigmoid + batch gradient descent.
class LogisticRegressionScratch:
    def __init__(self, lr=0.1, n_iters=2000):
        self.lr = lr
        self.n_iters = n_iters
        self.weights = None
        self.bias = None
        self.loss_history = []

    def _sigmoid(self, z):
        z = np.clip(z, -500, 500)  # numerical stability
        return 1 / (1 + np.exp(-z))

    def fit(self, X, y):
        X = np.array(X, dtype=float)
        y = np.array(y, dtype=float)
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0.0

        for _ in range(self.n_iters):
            linear = np.dot(X, self.weights) + self.bias
            preds = self._sigmoid(linear)

            # Gradients of the binary cross-entropy loss
            dw = (1 / n_samples) * np.dot(X.T, (preds - y))
            db = (1 / n_samples) * np.sum(preds - y)

            self.weights -= self.lr * dw
            self.bias -= self.lr * db

            eps = 1e-9
            loss = -np.mean(y * np.log(preds + eps) + (1 - y) * np.log(1 - preds + eps))
            self.loss_history.append(loss)

    def predict_proba(self, X):
        X = np.array(X, dtype=float)
        return self._sigmoid(np.dot(X, self.weights) + self.bias)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)


lr_scratch = LogisticRegressionScratch(lr=0.1, n_iters=3000)
lr_scratch.fit(X_train_scaled, y_train.values)

y_pred_scratch = lr_scratch.predict(X_test_scaled)
print('Scratch Logistic Regression - Test Accuracy:', round(accuracy_score(y_test, y_pred_scratch), 4))

In [ ]:
# Training loss curve of the from-scratch model (should decrease and flatten)
plt.figure(figsize=(7, 4))
plt.plot(lr_scratch.loss_history)
plt.xlabel('Iteration')
plt.ylabel('Binary Cross-Entropy Loss')
plt.title('From-Scratch Logistic Regression - Training Loss')
plt.show()

# Sanity check: library vs scratch should agree closely
print('Agreement between library and scratch predictions:',
      round(np.mean(y_pred_lib == y_pred_scratch), 4))

## Task 4: Test the Model and Compute Metrics

Evaluate the chosen model on the held-out test set with multiple metrics, inspect the confusion matrix, and check for overfitting / underfitting by comparing training and test performance.

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report,
                             roc_auc_score)

# Use the library Logistic Regression as the primary model
y_pred = lr_lib.predict(X_test_scaled)
y_proba = lr_lib.predict_proba(X_test_scaled)[:, 1]

print('Accuracy :', round(accuracy_score(y_test, y_pred), 4))
print('Precision:', round(precision_score(y_test, y_pred), 4))
print('Recall   :', round(recall_score(y_test, y_pred), 4))
print('F1 Score :', round(f1_score(y_test, y_pred), 4))
print('ROC AUC  :', round(roc_auc_score(y_test, y_proba), 4))
print('\nClassification Report:\n')
print(classification_report(y_test, y_pred, target_names=['Non Diabetic', 'Diabetic']))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Non Diabetic', 'Diabetic'],
            yticklabels=['Non Diabetic', 'Diabetic'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# --- Overfitting / Underfitting check ---
train_acc = accuracy_score(y_train, lr_lib.predict(X_train_scaled))
test_acc = accuracy_score(y_test, lr_lib.predict(X_test_scaled))

print('Training Accuracy:', round(train_acc, 4))
print('Test Accuracy    :', round(test_acc, 4))
print('Gap (train - test):', round(train_acc - test_acc, 4))

print('\nInterpretation:')
if train_acc - test_acc > 0.1:
    print('- Large gap -> the model is likely OVERFITTING.')
elif train_acc < 0.7 and test_acc < 0.7:
    print('- Both scores are low -> the model is likely UNDERFITTING.')
else:
    print('- Train and test scores are close and reasonable -> the model generalises well (good fit).')

## Task 5: Advanced Models, Cross-Validation and Hyperparameter Tuning

Experiment with several advanced models, use cross-validation to check stability, compare them on validation metrics, and tune the hyperparameters of the best model.

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold, GridSearchCV

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'SVM': SVC(probability=True, random_state=42),
    'KNN': KNeighborsClassifier()
}

In [ ]:
# --- Cross-validation to ensure model stability ---
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = []
for name, model in models.items():
    scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='accuracy')
    results.append({
        'Model': name,
        'CV Mean Accuracy': round(scores.mean(), 4),
        'CV Std': round(scores.std(), 4)
    })

results_df = pd.DataFrame(results).sort_values('CV Mean Accuracy', ascending=False)
results_df

In [ ]:
# --- Compare models on the test set (validation metrics) ---
comparison = []
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    comparison.append({
        'Model': name,
        'Accuracy': round(accuracy_score(y_test, preds), 4),
        'Precision': round(precision_score(y_test, preds), 4),
        'Recall': round(recall_score(y_test, preds), 4),
        'F1': round(f1_score(y_test, preds), 4)
    })

comparison_df = pd.DataFrame(comparison).sort_values('F1', ascending=False)
comparison_df

In [ ]:
# Visual comparison of test accuracy
plt.figure(figsize=(8, 4))
sns.barplot(x='Accuracy', y='Model', data=comparison_df)
plt.title('Model Comparison - Test Accuracy')
plt.xlim(0, 1)
plt.show()

In [ ]:
# --- Hyperparameter Tuning (GridSearchCV) on Random Forest ---
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=cv,
    scoring='f1',
    n_jobs=-1
)
grid.fit(X_train_scaled, y_train)

print('Best parameters:', grid.best_params_)
print('Best CV F1 score:', round(grid.best_score_, 4))

In [ ]:
# Evaluate the tuned model on the test set
best_model = grid.best_estimator_
best_pred = best_model.predict(X_test_scaled)

print('Tuned Random Forest - Test metrics')
print('Accuracy :', round(accuracy_score(y_test, best_pred), 4))
print('Precision:', round(precision_score(y_test, best_pred), 4))
print('Recall   :', round(recall_score(y_test, best_pred), 4))
print('F1 Score :', round(f1_score(y_test, best_pred), 4))

# Overfitting check for the tuned model
print('\nTrain Accuracy:', round(accuracy_score(y_train, best_model.predict(X_train_scaled)), 4))
print('Test Accuracy :', round(accuracy_score(y_test, best_pred), 4))

### Summary

- **Task 1-2:** Loaded and understood the data, replaced disguised (zero) missing values with the median, capped outliers with the IQR rule, encoded the `Outcome` target to 0/1, scaled the features, and explored distributions and correlations.
- **Task 3:** Selected Logistic Regression and implemented it both with scikit-learn and fully from scratch (NumPy gradient descent); the two implementations agree closely.
- **Task 4:** Evaluated accuracy, precision, recall, F1 and ROC AUC, inspected the confusion matrix, and confirmed the model is a good fit (small train/test gap).
- **Task 5:** Compared five models with 5-fold stratified cross-validation, ranked them on validation metrics, and tuned the Random Forest with GridSearchCV.